In [26]:
# check install for not common libraries 
!pip install databento


In [27]:
import databento as db 
import pandas as pd 
import numpy as np 



In [28]:
def fetch_databento_history(ticker_symbol, start_date="2019-01-01", end_date="2024-12-31"):
    print(f"📡 Requesting 2019-2024 15m data for {ticker_symbol} from Databento...")
    
    # Initialize client (You use your free tier API key here)
    client = db.Historical("db-pYirckELnpA3gjs3Rysns8xdFjRrU")
    
    # Request the 15-minute aggregated data
    # We use XNAS.ITCH (Nasdaq) or GLBX.MDP3 (CME) depending on the asset class
    data = client.timeseries.get_range(
        dataset='XNAS.ITCH', # Nasdaq Equities dataset
        symbols=ticker_symbol,
        schema='ohlcv-15m',  # Pre-calculated 15-minute intervals
        start=start_date,
        end=end_date,
    )
    
    # Convert directly to a Pandas DataFrame
    df = data.to_df()
    
    # Databento index is UTC, convert to US Eastern Market Hours
    df.index = df.index.tz_convert('America/New_York')
    
    # Filter out pre-market and after-hours trading (Optional, but highly recommended for this strategy)
    df = df.between_time('09:30', '16:00')
    
    # --- APPLYING OUR QUANT MATH TO DATABENTO ---
    df['Typical_Price'] = (df['high'] + df['low'] + df['close']) / 3
    df['Price_Volume'] = df['Typical_Price'] * df['volume']
    
    df['Date'] = df.index.date
    df['Cumulative_PV'] = df.groupby('Date')['Price_Volume'].cumsum()
    df['Cumulative_Volume'] = df.groupby('Date')['volume'].cumsum()
    
    df['VWAP'] = df['Cumulative_PV'] / df['Cumulative_Volume']
    df['VWAP_Distance_Pct'] = round(((df['close'] - df['VWAP']) / df['VWAP']) * 100, 3)
    
    # Rolling Volatility (Z-Score over 26 periods / 1 day)
    df['Return'] = df['close'].pct_change()
    df['Z_Score'] = round((df['Return'] - df['Return'].rolling(26).mean()) / df['Return'].rolling(26).std(), 3)
    
    # Volume Shock Ratio
    df['Volume_Shock'] = round(df['volume'] / (df['volume'].rolling(26).mean() + 1e-9), 2)
    
    df = df.dropna()
    
    # Keep only the pure numerical signals for the AI / Backtester
    df_clean = df[['close', 'volume', 'VWAP', 'VWAP_Distance_Pct', 'Z_Score', 'Volume_Shock']].rename(columns={'close': 'Price'})
    
    print(f"✅ Successfully loaded {len(df_clean)} historical intervals.")
    return df_clean

# Usage:
# historical_data = fetch_databento_history("BRK.B")

In [29]:
'''

historical_data = fetch_databento_history("BRK.B", start_date="2024-01-01", end_date="2024-01-10")

display(historical_data.head(10))

'''

'\n\nhistorical_data = fetch_databento_history("BRK.B", start_date="2024-01-01", end_date="2024-01-10")\n\ndisplay(historical_data.head(10))\n\n'